# Audio Unimodal Emotion Recognition (MELD)

Two CNN variants on the MELD dataset using Log-Mel Spectrograms:
- **`AudioCNNLight`** — fast, low-parameter, suitable for edge/real-time use
- **`AudioCNNHeavy`** — deeper, higher accuracy, for offline/server use

Select which to train by setting `MODEL_VARIANT` in the config cell.

## 1. Imports & Config

In [1]:
import warnings
from collections import Counter
from pathlib import Path

import librosa
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT = Path("../Data/MELD.Audio")
CSV_ROOT  = Path("../Data/MELD.Raw")
MODEL_DIR = Path("../Models/Audio_Models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Audio config ─────────────────────────────────────────────────────────────
SAMPLE_RATE = 16_000
MAX_LEN     = SAMPLE_RATE * 6    # 6 seconds

# ── Label config ─────────────────────────────────────────────────────────────
EMO_MAP     = {"joy": "happiness"}
LABELS      = ["anger", "disgust", "fear", "happiness", "sadness", "surprise", "neutral"]
NUM_CLASSES = len(LABELS)
LABEL2IDX   = {l: i for i, l in enumerate(LABELS)}

# ── Training config ──────────────────────────────────────────────────────────
# Set to "light" or "heavy" to choose which model to train
MODEL_VARIANT = "light"

BATCH_SIZE = 32
EPOCHS     = 50
LR         = 5e-4
PATIENCE   = 10    # early-stopping patience (epochs without macro-F1 improvement)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  Variant: {MODEL_VARIANT}")

Device: cuda  |  Variant: light


## 2. Feature Extraction

In [2]:
# n_mels=64 avoids the all-zero filterbank warning that occurs with
# n_mels=128 when n_fft=400 gives only 201 frequency bins.
mel_transform   = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=400, hop_length=160, n_mels=64
)
amplitude_to_db = torchaudio.transforms.AmplitudeToDB()


def fix_length(waveform: torch.Tensor) -> torch.Tensor:
    """Truncate or zero-pad waveform [1, T] to exactly MAX_LEN samples."""
    T = waveform.shape[1]
    if T > MAX_LEN:
        return waveform[:, :MAX_LEN]
    if T < MAX_LEN:
        return F.pad(waveform, (0, MAX_LEN - T))
    return waveform


def load_audio(path) -> torch.Tensor:
    """Load, resample and fix-length an audio file. Returns [1, MAX_LEN]."""
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return fix_length(torch.from_numpy(y).float().unsqueeze(0))


def extract_features(waveform: torch.Tensor) -> torch.Tensor:
    """Waveform [1, T]  ->  log-mel spectrogram [n_mels, time]."""
    return amplitude_to_db(mel_transform(waveform)).squeeze(0)

## 3. Dataset

In [3]:
CSV_PATHS = {
    "train": CSV_ROOT / "train" / "train_sent_emo.csv",
    "dev":   CSV_ROOT / "dev_sent_emo.csv",
    "test":  CSV_ROOT / "test_sent_emo.csv",
}


class MELDAudioDataset(Dataset):
    def __init__(self, split: str = "train"):
        assert split in CSV_PATHS, f"split must be one of {list(CSV_PATHS)}"
        audio_dir = DATA_ROOT / split

        df = pd.read_csv(CSV_PATHS[split])
        df["Emotion"] = df["Emotion"].str.lower().replace(EMO_MAP)

        # Build paths vectorised (avoids slow iterrows)
        df["filepath"] = df.apply(
            lambda r: audio_dir / f"dia{r.Dialogue_ID}_utt{r.Utterance_ID}.wav",
            axis=1,
        )

        exists_mask = df["filepath"].apply(lambda p: p.exists())
        if (~exists_mask).sum():
            warnings.warn(f"[{split}] {(~exists_mask).sum()} audio files missing — skipped.")

        known_mask = df["Emotion"].isin(LABEL2IDX)
        if (~known_mask).sum():
            warnings.warn(f"[{split}] {(~known_mask).sum()} unknown labels — skipped.")

        df = df[exists_mask & known_mask].reset_index(drop=True)

        self.filepaths = df["filepath"].tolist()
        self.labels    = df["Emotion"].map(LABEL2IDX).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        waveform = load_audio(self.filepaths[idx])
        return extract_features(waveform), self.labels[idx]


train_ds = MELDAudioDataset("train")
dev_ds   = MELDAudioDataset("dev")
x, y = train_ds[0]
print(f"Train: {len(train_ds)} | Dev: {len(dev_ds)}")
print(f"Feature shape: {x.shape}  |  Label: {y} ({LABELS[y]})")

C:\Users\Andrei\AppData\Local\Temp\ipykernel_20304\2348715303.py:24: UserWarning: [train] 1 audio files missing — skipped.
  warnings.warn(f"[{split}] {(~exists_mask).sum()} audio files missing — skipped.")
C:\Users\Andrei\AppData\Local\Temp\ipykernel_20304\2348715303.py:24: UserWarning: [dev] 1 audio files missing — skipped.
  warnings.warn(f"[{split}] {(~exists_mask).sum()} audio files missing — skipped.")
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 9988 | Dev: 1108
Feature shape: torch.Size([64, 601])  |  Label: 6 (neutral)


## 4. Class-Balanced Data Loaders

In [4]:
def make_weighted_sampler(dataset: MELDAudioDataset) -> WeightedRandomSampler:
    counts = Counter(dataset.labels)
    weights = [1.0 / (counts[lbl] ** 0.5) for lbl in dataset.labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


def make_class_weights(dataset: MELDAudioDataset, device) -> torch.Tensor:
    counts   = Counter(dataset.labels)
    counts_t = torch.tensor([counts[i] for i in range(NUM_CLASSES)], dtype=torch.float32)
    w = 1.0 / counts_t.sqrt()
    return (w / w.mean()).to(device)


sampler       = make_weighted_sampler(train_ds)
class_weights = make_class_weights(train_ds, DEVICE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=0)

print("Class weights:", class_weights.cpu().numpy().round(3))

Class weights: [0.816 1.652 1.661 0.651 1.04  0.783 0.396]


## 5. Models

### 5a. Lightweight model — fast & energy-efficient

Design goals: minimal multiply-accumulate operations, small memory footprint, suitable for real-time or on-device inference.

- Only **2 conv layers** (16 -> 32 channels)
- **Depthwise-separable convolutions** — same receptive field as standard Conv2d but ~8-9x fewer MACs
- **Global average pooling** into a single small linear head (no large FC layers)
- Dropout 0.2 (lighter regularisation)

In [5]:
class AudioCNNLight(nn.Module):
    """
    Lightweight audio CNN using depthwise-separable convolutions.
    ~35 K parameters. Suitable for edge / real-time inference.
    """

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()

        def ds_block(in_ch, out_ch):
            """Depthwise conv -> pointwise conv -> BN -> ReLU -> pool."""
            return nn.Sequential(
                nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1, groups=in_ch, bias=False),
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.encoder     = nn.Sequential(ds_block(1, 16), ds_block(16, 32))
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout     = nn.Dropout(0.2)
        self.classifier  = nn.Linear(32, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)       # [B, 1, n_mels, time]
        x = self.encoder(x)      # [B, 32, H', W']
        x = self.global_pool(x)  # [B, 32, 1, 1]
        x = x.flatten(1)         # [B, 32]
        x = self.dropout(x)
        return self.classifier(x)


_m = AudioCNNLight()
_x = torch.randn(2, 64, 601)
assert _m(_x).shape == (2, NUM_CLASSES)
print(f"AudioCNNLight  params: {sum(p.numel() for p in _m.parameters()):,}")
del _m, _x

AudioCNNLight  params: 1,008


### 5b. Heavy model — higher accuracy

Design goals: maximise representation power at the cost of training time.

- **4 conv layers** (32 -> 64 -> 128 -> 256 channels)
- Standard Conv2d + BatchNorm for the first two blocks
- **Residual skip connections** on the last two blocks to ease optimisation of the deeper stack
- Dropout 0.4 (stronger regularisation to counter higher capacity)

In [6]:
class _ResBlock(nn.Module):
    """Conv-BN-ReLU x2 with a residual skip connection."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.pool  = nn.MaxPool2d(2)
        # 1x1 projection if channel count changes
        self.skip  = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = F.relu(out + self.skip(x), inplace=True)
        return self.pool(out)


class AudioCNNHeavy(nn.Module):
    """
    Deep audio CNN with residual connections.
    ~1.5 M parameters. Optimised for accuracy over speed.
    """

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.encoder = nn.Sequential(
            conv_block(1,  32),
            conv_block(32, 64),
            _ResBlock(64,  128),
            _ResBlock(128, 256),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout     = nn.Dropout(0.4)
        self.classifier  = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)       # [B, 1, n_mels, time]
        x = self.encoder(x)      # [B, 256, H', W']
        x = self.global_pool(x)  # [B, 256, 1, 1]
        x = x.flatten(1)         # [B, 256]
        x = self.dropout(x)
        return self.classifier(x)


_m = AudioCNNHeavy()
_x = torch.randn(2, 64, 601)
assert _m(_x).shape == (2, NUM_CLASSES)
print(f"AudioCNNHeavy  params: {sum(p.numel() for p in _m.parameters()):,}")
del _m, _x

AudioCNNHeavy  params: 1,169,895


## 6. Training Utilities

In [7]:
def train_one_epoch(model, loader, optimizer, criterion, device) -> float:
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    """Returns (accuracy, macro_f1, classification_report_str)."""
    model.eval()
    all_preds, all_labels = [], []
    for x, y in loader:
        preds = model(x.to(device)).argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

    acc      = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    report   = classification_report(
        all_labels, all_preds, target_names=LABELS, digits=4, zero_division=0
    )
    return acc, macro_f1, report

## 7. Train

Change `MODEL_VARIANT` in the config cell (section 1) to switch between `"light"` and `"heavy"`.

In [ ]:
assert MODEL_VARIANT in ("light", "heavy"), "MODEL_VARIANT must be 'light' or 'heavy'"

model     = (AudioCNNLight() if MODEL_VARIANT == "light" else AudioCNNHeavy()).to(DEVICE)
save_path = MODEL_DIR / f"best_audio_{MODEL_VARIANT}.pth"

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_f1, patience_left = 0.0, PATIENCE

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_acc, val_f1, _ = evaluate(model, dev_loader, DEVICE)

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1, patience_left = val_f1, PATIENCE
        torch.save(model.state_dict(), save_path)
        flag = "  checkpointed"
    else:
        patience_left -= 1
        flag = f"  (patience {patience_left}/{PATIENCE})"

    print(
        f"Epoch {epoch:3d}/{EPOCHS}  "
        f"loss={train_loss:.4f}  "
        f"val_acc={val_acc:.4f}  "
        f"val_macro_f1={val_f1:.4f}{flag}"
    )

    if patience_left == 0:
        print(f"\nEarly stopping after {epoch} epochs.")
        break

print(f"\nBest macro-F1: {best_f1:.4f}  ->  {save_path}")

Epoch   1/50  loss=1.9496  val_acc=0.1588  val_macro_f1=0.1055  checkpointed
Epoch   2/50  loss=1.9431  val_acc=0.1345  val_macro_f1=0.0897  (patience 9/10)
Epoch   3/50  loss=1.9375  val_acc=0.1922  val_macro_f1=0.1443  checkpointed
Epoch   4/50  loss=1.9343  val_acc=0.1787  val_macro_f1=0.1291  (patience 9/10)
Epoch   5/50  loss=1.9277  val_acc=0.1841  val_macro_f1=0.1284  (patience 8/10)
Epoch   6/50  loss=1.9346  val_acc=0.1534  val_macro_f1=0.1079  (patience 7/10)
Epoch   7/50  loss=1.9347  val_acc=0.2049  val_macro_f1=0.1545  checkpointed
Epoch   8/50  loss=1.9293  val_acc=0.1507  val_macro_f1=0.1062  (patience 9/10)
Epoch   9/50  loss=1.9347  val_acc=0.1715  val_macro_f1=0.1202  (patience 8/10)
Epoch  10/50  loss=1.9254  val_acc=0.1597  val_macro_f1=0.1233  (patience 7/10)
Epoch  11/50  loss=1.9298  val_acc=0.2274  val_macro_f1=0.1632  checkpointed


KeyboardInterrupt: 

: 

## 8. Final Evaluation

In [ ]:
best_model = (AudioCNNLight() if MODEL_VARIANT == "light" else AudioCNNHeavy()).to(DEVICE)
best_model.load_state_dict(torch.load(save_path, map_location=DEVICE))

acc, macro_f1, report = evaluate(best_model, dev_loader, DEVICE)
print(f"[{MODEL_VARIANT}]  Dev accuracy: {acc:.4f}  |  Dev macro-F1: {macro_f1:.4f}")
print()
print(report)

[heavy]  Dev accuracy: 0.3872  |  Dev macro-F1: 0.2337

              precision    recall  f1-score   support

       anger     0.3956    0.2353    0.2951       153
     disgust     0.0508    0.2727    0.0857        22
        fear     0.0000    0.0000    0.0000        40
   happiness     0.1933    0.1411    0.1631       163
     sadness     0.2817    0.3604    0.3162       111
    surprise     0.3088    0.1400    0.1927       150
     neutral     0.5316    0.6461    0.5833       469

    accuracy                         0.3872      1108
   macro avg     0.2517    0.2565    0.2337      1108
weighted avg     0.3791    0.3872    0.3711      1108

